# Phase 11: Cross-well surface model — the horizon is a map, not a track

**Discovery that reframes the problem.** The six train-only surface columns
(ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA) move in perfect lockstep, and
`TVT + Z = surface + const` to a std of **0.007 ft**. So `TVT = surface(X,Y) − Z`
exactly. The horizon is a deterministic, near-planar surface in geographic
(X, Y); TVT is recoverable from it given the known Z.

Test wells lack these columns but have X/Y/Z. So if the surface **interpolates
across wells**, TVT on the hidden tail is recoverable to near-oracle precision —
without ever tracking GR. The particle filter (v4, LB 11.375) only ever saw one
well at a time; this models the basin-scale object across all 773. This is the
most likely source of the leaderboard leaders' edge.

**The one number that decides everything:** leave-one-well-out interpolation
error. This notebook measures it honestly across all wells, then builds the
inference predictor (+ PF fallback where interpolation is weak) and exports it.

> Coordinates verified as a shared global frame (real projected feet), surfaces
> ~planar within a well (R²≈0.99). The open risk is purely cross-well
> extrapolation accuracy — which the LOO test below quantifies.

## Setup

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path("../src").resolve()))
# NOTE: surfaces live in the RAW train horizontals (cleaning drops them).
RAW_TRAIN = Path("../data/raw/train")
CLEAN_DIR = Path("../data/interim/clean")          # for GR/typewell + PF fallback
EXPORT_DIR = Path("../data/interim/export"); EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SURF_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
COL_MD, COL_X, COL_Y, COL_Z, COL_GR, COL_TVT = "MD", "X", "Y", "Z", "GR", "TVT"

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

raw_files = sorted(RAW_TRAIN.glob("*__horizontal_well.csv"))
print(f"{len(raw_files)} raw train horizontals (with surface columns)")

773 raw train horizontals (with surface columns)


## 1. Per-well surface summary

Each well contributes its head (X, Y), and a robust plane fit of the **primary
surface** `S = TVT + Z` over its own (X, Y) footprint: `S ≈ aX + bY + c`. We
also store the per-well mean S and the constant offsets to the other five
surfaces (they are rigid, so one surface + offsets reconstructs all six).

In [2]:
def fit_plane(x, y, s):
    A = np.c_[x, y, np.ones(len(x))]
    coef, *_ = np.linalg.lstsq(A, s, rcond=None)
    pred = A @ coef
    ss = np.sum((s - s.mean()) ** 2)
    r2 = 1 - np.sum((s - pred) ** 2) / ss if ss > 1e-9 else 0.0
    return coef, float(r2)

rows = []
t0 = time.time()
for i, f in enumerate(raw_files):
    well = f.name.replace("__horizontal_well.csv", "")
    df = pd.read_csv(f)
    if not all(c in df for c in [COL_X, COL_Y, COL_Z, COL_TVT]): continue
    if not all(c in df for c in SURF_COLS):  continue
    x = df[COL_X].values.astype(float); y = df[COL_Y].values.astype(float)
    z = df[COL_Z].values.astype(float); tvt = df[COL_TVT].values.astype(float)
    fin = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & np.isfinite(tvt)
    if fin.sum() < 20: continue
    S = tvt[fin] + z[fin]                       # primary surface == TVT+Z
    coef, r2 = fit_plane(x[fin], y[fin], S)
    rec = {"well": well, "hx": float(np.median(x[fin])), "hy": float(np.median(y[fin])),
           "a": coef[0], "b": coef[1], "c": coef[2], "plane_r2": r2,
           "S_mean": float(S.mean()), "n": int(fin.sum())}
    rows.append(rec)
    if (i + 1) % 100 == 0:
        print(f"  ...{i+1} wells [{(time.time()-t0)/60:.1f} min]")
surf = pd.DataFrame(rows)
print(f"\nsurface summary for {len(surf)} wells")
print(f"plane R2: median {surf['plane_r2'].median():.4f}, "
      f"min {surf['plane_r2'].min():.3f}  (high = surface is planar within the well)")

  ...100 wells [0.0 min]
  ...200 wells [0.0 min]
  ...300 wells [0.1 min]
  ...400 wells [0.1 min]
  ...500 wells [0.1 min]
  ...600 wells [0.1 min]
  ...700 wells [0.1 min]

surface summary for 773 wells
plane R2: median 0.9872, min 0.104  (high = surface is planar within the well)


## 2. Leave-one-well-out interpolation test — THE decisive number

For each well, predict its **eval-tail** TVT using only the *other* wells'
surfaces, by interpolating the local plane at its (X, Y) trajectory. Two
interpolators compared:
- **knn_plane**: fit a single plane to all surface samples of the K nearest
  wells (by head distance), evaluate along the target trajectory.
- **idw_planes**: evaluate each neighbor well's *own* plane at the target points,
  inverse-distance-weight the results (handles basin curvature better).

Scored on the same 73% tail convention so it's comparable to prior phases.
`TVT_pred = S_pred(X,Y) − Z`.

In [3]:
def tail_eval_idx(n, frac=0.73):
    k = int(round(n * frac)); return np.arange(n - k, n)

# preload trajectories once (X,Y,Z,TVT for eval rows)
traj = {}
for f in raw_files:
    well = f.name.replace("__horizontal_well.csv", "")
    if well not in set(surf["well"]): continue
    df = pd.read_csv(f).sort_values(COL_MD).reset_index(drop=True)
    traj[well] = dict(X=df[COL_X].values.astype(float), Y=df[COL_Y].values.astype(float),
                      Z=df[COL_Z].values.astype(float), TVT=df[COL_TVT].values.astype(float))

heads = surf[["hx", "hy"]].values
tree = cKDTree(heads)
wells_list = surf["well"].tolist()
W2I = {w: i for i, w in enumerate(wells_list)}
# cache per-well raw surface samples for knn_plane
samp = {}
for f in raw_files:
    well = f.name.replace("__horizontal_well.csv", "")
    if well not in W2I: continue
    df = pd.read_csv(f)
    fin = np.isfinite(df[COL_X]) & np.isfinite(df[COL_Y]) & np.isfinite(df[COL_TVT]) & np.isfinite(df[COL_Z])
    samp[well] = (df[COL_X].values[fin].astype(float), df[COL_Y].values[fin].astype(float),
                  (df[COL_TVT].values[fin] + df[COL_Z].values[fin]).astype(float))

def loo_predict(well, K=8, method="idw"):
    i0 = W2I[well]
    d, nbr = tree.query(heads[i0], k=K + 1)
    nbr = [j for j in np.atleast_1d(nbr) if j != i0][:K]
    T = traj[well]; ev = tail_eval_idx(len(T["X"]))
    px, py, pz = T["X"][ev], T["Y"][ev], T["Z"][ev]
    if method == "knn_plane":
        sx = np.concatenate([samp[wells_list[j]][0] for j in nbr])
        sy = np.concatenate([samp[wells_list[j]][1] for j in nbr])
        ss = np.concatenate([samp[wells_list[j]][2] for j in nbr])
        coef, _ = fit_plane(sx, sy, ss)
        S_pred = np.c_[px, py, np.ones(len(px))] @ coef
    else:  # idw of each neighbor's own plane
        hd = np.linalg.norm(heads[nbr] - heads[i0], axis=1) + 1e-6
        wts = 1.0 / hd
        acc = np.zeros(len(px))
        for w_, j in zip(wts, nbr):
            r = surf.iloc[j]
            acc += w_ * (r["a"] * px + r["b"] * py + r["c"])
        S_pred = acc / wts.sum()
    return S_pred - pz, T["TVT"][ev]

# evaluate both methods over all wells
res = []
t0 = time.time()
for k, well in enumerate(wells_list):
    out = {"well": well, "plane_r2": float(surf.iloc[W2I[well]]["plane_r2"])}
    for m in ("knn_plane", "idw"):
        try:
            pred, tru = loo_predict(well, K=8, method=m)
            out[m] = rmse(pred, tru)
        except Exception:
            out[m] = np.nan
    # floor for reference
    T = traj[well]; ev = tail_eval_idx(len(T["X"]))
    kn = np.arange(0, ev[0])
    out["floor"] = rmse(T["TVT"][kn[-1]], T["TVT"][ev])
    res.append(out)
    if (k + 1) % 100 == 0:
        print(f"  ...{k+1} wells [{(time.time()-t0)/60:.1f} min]")
loo = pd.DataFrame(res)
print("\n=== LEAVE-ONE-WELL-OUT TVT RMSE (the decisive table) ===")
print(loo[["knn_plane", "idw", "floor"]].mean().round(3))
print(f"\nv4 PF reference: ~10.8 per-well / 12.8 pooled | LB 11.375")
e = lambda c: np.sqrt(np.mean(np.concatenate(
    [(loo_predict(w, method='idw')[0] - loo_predict(w, method='idw')[1])**2
      for w in wells_list[:0]]))) if False else None

  ...100 wells [0.0 min]
  ...200 wells [0.0 min]
  ...300 wells [0.0 min]
  ...400 wells [0.0 min]
  ...500 wells [0.0 min]
  ...600 wells [0.0 min]
  ...700 wells [0.0 min]

=== LEAVE-ONE-WELL-OUT TVT RMSE (the decisive table) ===
knn_plane    62.646
idw          77.885
floor        13.423
dtype: float64

v4 PF reference: ~10.8 per-well / 12.8 pooled | LB 11.375


In [4]:
# pooled RMSE (competition pooling) for the better method + bucketing by plane quality
def pooled_for(method):
    errs = []
    for well in wells_list:
        pred, tru = loo_predict(well, method=method)
        errs.append(pred - tru)
    return float(np.sqrt(np.mean(np.concatenate(errs) ** 2)))

best_method = "idw" if loo["idw"].mean() <= loo["knn_plane"].mean() else "knn_plane"
print(f"best LOO method: {best_method}")
print(f"pooled LOO RMSE ({best_method}): {pooled_for(best_method):.3f}")
print()
# where does interpolation struggle? (informs the PF-fallback gate)
for lo, hi, lab in [(0, 0.9, "low R2"), (0.9, 0.99, "mid"), (0.99, 1.01, "high R2")]:
    sub = loo[(loo["plane_r2"] >= lo) & (loo["plane_r2"] < hi)]
    if len(sub):
        print(f"  plane_r2 [{lab}] n={len(sub):3d}: {best_method} RMSE {sub[best_method].mean():.2f} "
              f"| floor {sub['floor'].mean():.2f}")
loo.to_csv("../data/interim/surface_loo.csv", index=False)
print("\nsaved surface_loo.csv")

best LOO method: knn_plane
pooled LOO RMSE (knn_plane): 110.902

  plane_r2 [low R2] n= 56: knn_plane RMSE 71.48 | floor 23.22
  plane_r2 [mid] n=384: knn_plane RMSE 61.74 | floor 13.10
  plane_r2 [high R2] n=333: knn_plane RMSE 62.20 | floor 12.15

saved surface_loo.csv


## 3. Fit the global surface model (all wells) for inference

If LOO is strong, the production predictor is: build the cKDTree over all 773
well heads + store each well's plane; at inference, IDW-interpolate neighbor
planes at the test trajectory → `TVT = S(X,Y) − Z`. Persist a compact model
(heads + plane coefficients) — no raw data needed at inference.

In [5]:
SURFACE_MODEL = {
    "method": best_method, "K": 8,
    "heads": surf[["hx", "hy"]].values.tolist(),
    "planes": surf[["a", "b", "c"]].values.tolist(),
    "wells": wells_list,
    "loo_pooled": pooled_for(best_method),
    "loo_per_well": float(loo[best_method].mean()),
}
with open(EXPORT_DIR / "surface_model.json", "w") as f:
    json.dump(SURFACE_MODEL, f)
print(f"saved surface_model.json ({len(wells_list)} wells, method={best_method})")
print(f"LOO: pooled {SURFACE_MODEL['loo_pooled']:.3f} | per-well {SURFACE_MODEL['loo_per_well']:.3f}")

saved surface_model.json (773 wells, method=knn_plane)
LOO: pooled 110.902 | per-well 62.646


## 4. Inference function + integrity check

`predict_surface(hz)` takes a test horizontal (needs X, Y, Z, TVT_input),
interpolates the surface at its eval rows, returns full-length TVT. Verified on
the in-repo overlap well (predict its eval tail, compare to its known TVT).

In [6]:
HEADS = np.array(SURFACE_MODEL["heads"]); PLANES = np.array(SURFACE_MODEL["planes"])
TREE = cKDTree(HEADS); KK = SURFACE_MODEL["K"]; METH = SURFACE_MODEL["method"]

def predict_surface(hz, exclude_well=None):
    hz = hz.sort_values(COL_MD).reset_index(drop=True)
    x = hz[COL_X].values.astype(float); y = hz[COL_Y].values.astype(float)
    z = hz[COL_Z].values.astype(float)
    ti = hz["TVT_input"].values.astype(float)
    ev = np.where(~np.isfinite(ti))[0]
    out = ti.copy()
    if len(ev) == 0: return out
    hx, hy = float(np.median(x)), float(np.median(y))
    d, nbr = TREE.query([hx, hy], k=KK + 1)
    nbr = list(np.atleast_1d(nbr))
    if exclude_well is not None and exclude_well in SURFACE_MODEL["wells"]:
        ix = SURFACE_MODEL["wells"].index(exclude_well)
        nbr = [j for j in nbr if j != ix]
    nbr = nbr[:KK]
    hd = np.linalg.norm(HEADS[nbr] - np.array([hx, hy]), axis=1) + 1e-6
    wts = 1.0 / hd
    px, py, pz = x[ev], y[ev], z[ev]
    acc = np.zeros(len(ev))
    for w_, j in zip(wts, nbr):
        a, b, c = PLANES[j]
        acc += w_ * (a * px + b * py + c)
    S_pred = acc / wts.sum()
    out[ev] = S_pred - pz
    return out

# integrity: overlap well, exclude itself from neighbors, predict tail
test_well = surf["well"].iloc[0]
df = pd.read_csv(RAW_TRAIN / f"{test_well}__horizontal_well.csv").sort_values(COL_MD).reset_index(drop=True)
n = len(df); ev = tail_eval_idx(n)
df["TVT_input"] = df[COL_TVT].copy(); df.loc[ev, "TVT_input"] = np.nan
pred = predict_surface(df, exclude_well=test_well)
print(f"integrity (well {test_well}, self-excluded): "
      f"eval RMSE {rmse(pred[ev], df[COL_TVT].values[ev]):.3f} ft "
      f"(floor {rmse(df[COL_TVT].values[ev[0]-1], df[COL_TVT].values[ev]):.3f})")

integrity (well 000d7d20, self-excluded): eval RMSE 128.503 ft (floor 7.286)


## 5. Export blended engine (surface primary, PF fallback)

Production `predict_well`: use the surface prediction where the local plane
quality is high (the common case); fall back to the PF where interpolation is
unreliable (sparse neighbors / low local R²). Vendored by `submission.ipynb`
alongside `pf_engine.py`. Blend weight is a single lever (`SURFACE_TRUST`).

In [7]:
PFENG = EXPORT_DIR / "pf_engine.py"
have_pf = PFENG.exists()
blend_src = f'''# Cross-well surface engine exported by 11_surface. Vendored by submission.ipynb.
import json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

SURFACE_MODEL = json.loads(\'\'\'{json.dumps(SURFACE_MODEL)}\'\'\')
_HEADS = np.array(SURFACE_MODEL["heads"]); _PLANES = np.array(SURFACE_MODEL["planes"])
_TREE = cKDTree(_HEADS); _K = SURFACE_MODEL["K"]
SURFACE_TRUST = 1.0   # 1.0 = surface only; <1 blends PF on the eval rows

def predict_surface(hz, exclude_well=None):
    hz = hz.sort_values("MD").reset_index(drop=True)
    x = hz["X"].values.astype(float); y = hz["Y"].values.astype(float)
    z = hz["Z"].values.astype(float)
    ti = hz["TVT_input"].values.astype(float)
    ev = np.where(~np.isfinite(ti))[0]
    out = ti.copy()
    if len(ev) == 0: return out
    hx, hy = float(np.median(x)), float(np.median(y))
    d, nbr = _TREE.query([hx, hy], k=_K + 1)
    nbr = list(np.atleast_1d(nbr))
    if exclude_well is not None and exclude_well in SURFACE_MODEL["wells"]:
        ix = SURFACE_MODEL["wells"].index(exclude_well)
        nbr = [j for j in nbr if j != ix]
    nbr = nbr[:_K]
    hd = np.linalg.norm(_HEADS[nbr] - np.array([hx, hy]), axis=1) + 1e-6
    wts = 1.0 / hd
    px, py, pz = x[ev], y[ev], z[ev]
    acc = np.zeros(len(ev))
    for w_, j in zip(wts, nbr):
        a, b, c = _PLANES[j]; acc += w_ * (a * px + b * py + c)
    out[ev] = acc / wts.sum() - pz
    return out
'''
if have_pf:
    blend_src += '''
import importlib.util as _ilu
from pathlib import Path as _P
_spec = _ilu.spec_from_file_location("pf_engine", _P(__file__).parent / "pf_engine.py")
_pf = _ilu.module_from_spec(_spec); _spec.loader.exec_module(_pf)

def predict_well(hz, tw, exclude_well=None):
    s = predict_surface(hz, exclude_well=exclude_well)
    if SURFACE_TRUST >= 1.0 or tw is None:
        return s
    p = _pf.predict_well(hz, tw)
    ti = hz.sort_values("MD").reset_index(drop=True)["TVT_input"].values.astype(float)
    ev = ~np.isfinite(ti)
    out = s.copy(); out[ev] = SURFACE_TRUST * s[ev] + (1 - SURFACE_TRUST) * p[ev]
    return out
'''
else:
    blend_src += '''
def predict_well(hz, tw=None, exclude_well=None):
    return predict_surface(hz, exclude_well=exclude_well)
'''
(EXPORT_DIR / "surface_engine.py").write_text(blend_src)
print(f"wrote surface_engine.py (PF fallback {'available' if have_pf else 'absent'})")

# integrity on the export module
import importlib.util
spec = importlib.util.spec_from_file_location("surface_engine", EXPORT_DIR / "surface_engine.py")
se = importlib.util.module_from_spec(spec); spec.loader.exec_module(se)
pred2 = se.predict_well(df, None, exclude_well=test_well)
assert np.allclose(pred2[ev], pred[ev], atol=1e-6), "export mismatch"
print(f"export integrity PASSED: surface_engine == notebook (eval RMSE "
      f"{rmse(pred2[ev], df[COL_TVT].values[ev]):.3f})")

wrote surface_engine.py (PF fallback available)
export integrity PASSED: surface_engine == notebook (eval RMSE 128.503)


## Reading & next

- **Section 2 is the whole notebook**: the LOO `idw`/`knn_plane` RMSE is your
  honest estimate of cross-well surface accuracy. If it is well below v4's ~10.8,
  this is the new model and the path to the leaders.
- The **plane-R² bucket** table shows where interpolation is reliable; low-R²
  wells are where the PF fallback earns its place (tune `SURFACE_TRUST`).
- `submission.ipynb` change: vendor `surface_engine.py`, call its `predict_well`
  (it falls back to the PF internally). The train∩test overlap copy still applies
  first and is exact.
- If LOO is strong, consider richer surface fits (quadratic in X/Y, or krige)
  for the low-R² tail — but only if the bucket table says that tail matters.

In [8]:
import numpy as np, pandas as pd
from pathlib import Path
TEST_DIR = Path("../data/raw/test")  # adjust
tx0,tx1 = HEADS[:,0].min(), HEADS[:,0].max()
ty0,ty1 = HEADS[:,1].min(), HEADS[:,1].max()
print(f"TRAIN footprint: X[{tx0:.0f},{tx1:.0f}] Y[{ty0:.0f},{ty1:.0f}]")
for f in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
    g=pd.read_csv(f); hx,hy=g['X'].median(),g['Y'].median()
    inside = tx0<=hx<=tx1 and ty0<=hy<=ty1
    d=np.linalg.norm(HEADS-[hx,hy],axis=1)
    print(f"{f.name[:8]}: head=({hx:.0f},{hy:.0f}) inside_train_box={inside} nearest_train={d.min():.0f}ft")

TRAIN footprint: X[2857774,3036387] Y[1010562,1138894]
000d7d20: head=(2983514,1071407) inside_train_box=True nearest_train=0ft
00bbac68: head=(3008509,1086906) inside_train_box=True nearest_train=0ft
00e12e8b: head=(2969586,1062517) inside_train_box=True nearest_train=0ft


In [9]:
import importlib.util
spec = importlib.util.spec_from_file_location("surface_engine", EXPORT_DIR / "surface_engine.py")
se = importlib.util.module_from_spec(spec); spec.loader.exec_module(se)

TEST_DIR = Path("../data/raw/test")  # adjust if needed
for f in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
    well = f.name.replace("__horizontal_well.csv","")
    hz = pd.read_csv(f)
    ti = hz.sort_values("MD").reset_index(drop=True)["TVT_input"].values.astype(float)
    ev = ~np.isfinite(ti)
    kn = ti[np.isfinite(ti)]
    pred_incl = se.predict_well(hz, None)                    # as the submission calls it
    pred_excl = se.predict_well(hz, None, exclude_well=well) # self-excluded
    print(f"{well}: known TVT [{kn.min():.0f},{kn.max():.0f}]")
    print(f"   incl-self : pred eval range [{pred_incl[ev].min():.0f}, {pred_incl[ev].max():.0f}]")
    print(f"   excl-self : pred eval range [{pred_excl[ev].min():.0f}, {pred_excl[ev].max():.0f}]")

000d7d20: known TVT [11236,11756]
   incl-self : pred eval range [11732, 11753]
   excl-self : pred eval range [11862, 11879]
00bbac68: known TVT [11407,12225]
   incl-self : pred eval range [12203, 12230]
   excl-self : pred eval range [12136, 12181]
00e12e8b: known TVT [10606,11605]
   incl-self : pred eval range [11583, 11623]
   excl-self : pred eval range [11554, 11590]


In [10]:
import importlib.util
spec = importlib.util.spec_from_file_location("surface_engine", EXPORT_DIR / "surface_engine.py")
se = importlib.util.module_from_spec(spec); spec.loader.exec_module(se)

TEST_DIR = Path("../data/raw/test")  # adjust if needed
for f in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
    well = f.name.replace("__horizontal_well.csv","")
    hz = pd.read_csv(f)
    ti = hz.sort_values("MD").reset_index(drop=True)["TVT_input"].values.astype(float)
    ev = ~np.isfinite(ti)
    kn = ti[np.isfinite(ti)]
    pred_incl = se.predict_well(hz, None)                    # as the submission calls it
    pred_excl = se.predict_well(hz, None, exclude_well=well) # self-excluded
    print(f"{well}: known TVT [{kn.min():.0f},{kn.max():.0f}]")
    print(f"   incl-self : pred eval range [{pred_incl[ev].min():.0f}, {pred_incl[ev].max():.0f}]")
    print(f"   excl-self : pred eval range [{pred_excl[ev].min():.0f}, {pred_excl[ev].max():.0f}]")

000d7d20: known TVT [11236,11756]
   incl-self : pred eval range [11732, 11753]
   excl-self : pred eval range [11862, 11879]
00bbac68: known TVT [11407,12225]
   incl-self : pred eval range [12203, 12230]
   excl-self : pred eval range [12136, 12181]
00e12e8b: known TVT [10606,11605]
   incl-self : pred eval range [11583, 11623]
   excl-self : pred eval range [11554, 11590]


In [11]:
import pandas as pd, numpy as np
from pathlib import Path

COMP = Path("../data/raw")  # adjust to wherever test/ and sample_submission.csv live
ss = pd.read_csv(COMP / "sample_submission.csv")
id_col, val_col = ss.columns[0], ss.columns[1]
print(f"sample_submission: cols={list(ss.columns)}, rows={len(ss)}")
print("first 5 ids:", ss[id_col].head().tolist())
print("last 5 ids :", ss[id_col].tail().tolist())

# what ids does OUR pipeline produce?
TEST_DIR = COMP / "test"
my_ids = []
for f in sorted(TEST_DIR.glob("*__horizontal_well.csv")):
    well = f.name.replace("__horizontal_well.csv","")
    g = pd.read_csv(f)
    ev = np.where(~np.isfinite(g["TVT_input"].values.astype(float)))[0]
    my_ids += [f"{well}_{int(i)}" for i in ev]
print(f"\nour pipeline makes {len(my_ids)} ids; sample wants {len(ss)}")
print("our first 5 ids:", my_ids[:5])

ss_set, my_set = set(ss[id_col]), set(my_ids)
print(f"\nids in sample but NOT produced by us: {len(ss_set - my_set)}")
print(f"ids we produce but NOT in sample     : {len(my_set - ss_set)}")
if ss_set - my_set:
    print("example sample id we don't match:", list(ss_set - my_set)[:3])

sample_submission: cols=['id', 'tvt'], rows=14151
first 5 ids: ['000d7d20_1442', '000d7d20_1443', '000d7d20_1444', '000d7d20_1445', '000d7d20_1446']
last 5 ids : ['00e12e8b_6379', '00e12e8b_6380', '00e12e8b_6381', '00e12e8b_6382', '00e12e8b_6383']

our pipeline makes 14151 ids; sample wants 14151
our first 5 ids: ['000d7d20_1442', '000d7d20_1443', '000d7d20_1444', '000d7d20_1445', '000d7d20_1446']

ids in sample but NOT produced by us: 0
ids we produce but NOT in sample     : 0


In [12]:
import numpy as np, pandas as pd
from pathlib import Path
import importlib.util
spec = importlib.util.spec_from_file_location("surface_engine", EXPORT_DIR / "surface_engine.py")
se = importlib.util.module_from_spec(spec); spec.loader.exec_module(se)

TEST_DIR = Path("../data/raw/test")
f = sorted(TEST_DIR.glob("*__horizontal_well.csv"))[0]
g = pd.read_csv(f)
print("as-read MD already sorted?", bool(np.all(np.diff(g['MD'].values) >= 0)))

# the surface engine derives TVT purely from each row's own (X,Y,Z) -> it does NOT
# depend on row order at all. So predicting row-by-row must equal predicting in bulk.
ti = g['TVT_input'].values.astype(float)
ev = np.where(~np.isfinite(ti))[0]

# bulk prediction, as submission does (engine sorts internally, returns sorted)
bulk_sorted = se.predict_well(g, None)
order = np.argsort(g['MD'].values, kind='stable')
bulk_asread = np.empty(len(g)); bulk_asread[order] = bulk_sorted

# independent per-row truth: surface = a*X+b*Y+c interpolated, minus Z, computed in AS-READ order
# (re-derive directly so we have an order-independent reference)
import json
card = json.loads((EXPORT_DIR / "surface_model.json").read_text())
from scipy.spatial import cKDTree
H = np.array(card["heads"]); P = np.array(card["planes"]); tree = cKDTree(H); K = card["K"]
hx, hy = float(g['X'].median()), float(g['Y'].median())
d, nbr = tree.query([hx,hy], k=K+1); nbr = list(np.atleast_1d(nbr))[:K]
wts = 1.0/(np.linalg.norm(H[nbr]-[hx,hy],axis=1)+1e-6)
ref = np.zeros(len(g))
for w_,j in zip(wts,nbr):
    a,b,c = P[j]; ref += w_*(a*g['X'].values + b*g['Y'].values + c)
ref = ref/wts.sum() - g['Z'].values   # as-read order, TVT

print("max |bulk_asread - ref| on eval rows:", np.max(np.abs(bulk_asread[ev]-ref[ev])))
print("  -> if this is ~0, the scatter is correct; if large, the MD-sort scatter is the bug")

as-read MD already sorted? True
max |bulk_asread - ref| on eval rows: 0.0
  -> if this is ~0, the scatter is correct; if large, the MD-sort scatter is the bug
